# Metrics CERCA

In [1]:
import pandas as pd
import gender_guesser.detector as gender
from df2gspread import gspread2df as g2d

from tqdm import tqdm
tqdm.pandas()

In [2]:
interest_centers = ['ISGlobal']
file_path = '../data/external/5_Bibliometria_SIRIS_081025/'
center_name = 'ISGlobal/'
file_name = 'ISGlobal_DOIs Publications_2021-2024'

df_whole = []
for year in range(2021, 2025):
    df = pd.read_excel(file_path + 'BM_' + center_name + file_name + '.xlsx', sheet_name = str(year), skiprows = 1)
    df_whole.append(df[['Doi']])
df_whole = pd.concat(df_whole, ignore_index=True).rename(columns = {'Doi' : 'DOI'} )
df_whole['Center'] = 'ISGlobal'

df = pd.read_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_OA_ISGlobal.csv')
df

/tmp/ipykernel_7367/302042547.py:13: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_OA_ISGlobal.csv')


,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center
0,10.1289/ehp8117,Yoav Levi,7.0,middle,False,4.210100e+09,IL,False,ISGlobal
1,10.1038/s41588-021-00786-2,Tomislav Kuliš,192.0,middle,False,1.389377e+08,HR,False,ISGlobal
2,10.1038/s41588-021-00786-2,Tomislav Kuliš,192.0,middle,False,1.813434e+08,HR,False,ISGlobal
3,10.1111/all.14422,Mario Sánchez‐Borges,249.0,middle,False,4.210141e+09,VE,False,ISGlobal
4,10.1186/s13054-024-05180-y,Cristian C. Serrano-Mayorga,2.0,middle,False,1.576505e+08,CO,False,ISGlobal
...,...,...,...,...,...,...,...,...,...
85978,10.1038/s41467-021-24673-w,Linda‐Gail Bekker,7.0,middle,False,1.380251e+08,ZA,False,ISGlobal
85979,10.1038/s41380-020-00976-0,Nastassja Koen,17.0,middle,False,1.576143e+08,ZA,False,ISGlobal
85980,10.1371/journal.pone.0248538,Fredros O. Okumu,8.0,last,False,1.926191e+08,ZA,False,ISGlobal
85981,10.1038/s41572-023-00452-3,Linda‐Gail Bekker,1.0,first,True,1.576143e+08,ZA,False,ISGlobal


In [3]:
df_whole.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
ISGlobal    2744
dtype: int64

In [4]:
df[df.CERCA == True].drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
ISGlobal    2240
dtype: int64

## Publications Number

In [5]:
print('The total percentage of publications analyzed is:', df.DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.9183673469387755


In [6]:
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
ISGlobal    2520
dtype: int64

## % led publications

In [7]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.8163265306122449


**Percentage computed with the retrieved OA-raw_affiliation data identified author and not the whole**

In [8]:
df_cerca = df[df.CERCA == True]
df_led = df_cerca[(df_cerca.author_position == 'first') | (df_cerca.author_position == 'last') |(df_cerca.is_corresponding == True) ]
df_led.drop_duplicates(['DOI', 'Center']).groupby('Center').size() / \
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
ISGlobal    0.470238
dtype: float64

## \% publications with women from the centre as authors

In [9]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.8163265306122449


**Percentage computed with the retrieved OA-raw_affiliation data identified author and not the whole**

In [10]:
gend = gender.Detector()

df_cerca = df[df.CERCA == True].dropna(subset = 'display_name') # TO DELETE NON FOUND AUTHORS

df_cerca['first_name'] = df_cerca['display_name'].str.split(' ').str[0]
df_cerca['gender'] = df_cerca.first_name.progress_apply(lambda x: gend.get_gender(x))

df_cerca.drop_duplicates('first_name').sort_values('first_name', ascending = False).to_csv('gender_check_ISGlobal.csv', index = False)
df_cerca

100%|██████████| 16859/16859 [00:00<00:00, 359900.30it/s]


,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center,first_name,gender
143,10.1136/bmjopen-2021-051278,Carlos Chaccour,12.0,last,False,1.142924e+08,TZ,True,ISGlobal,Carlos,male
163,10.3389/fphar.2021.625678,Carlos Chaccour,3.0,middle,False,1.142924e+08,TZ,True,ISGlobal,Carlos,male
168,10.1371/journal.pntd.0009144,Carlos Chaccour,6.0,middle,True,1.142924e+08,TZ,True,ISGlobal,Carlos,male
190,10.4269/ajtmh.20-1531,Carlos Chaccour,3.0,last,False,1.142924e+08,TZ,True,ISGlobal,Carlos,male
191,10.1098/rstb.2019.0810,Carlos Chaccour,1.0,first,True,1.142924e+08,TZ,True,ISGlobal,Carlos,male
...,...,...,...,...,...,...,...,...,...,...,...
85771,10.1002/ppul.25405,Elisa López‐Varela,15.0,last,False,1.380251e+08,ZA,True,ISGlobal,Elisa,female
85772,10.1002/ppul.25405,Elisa López‐Varela,15.0,last,False,2.609232e+07,ZA,True,ISGlobal,Elisa,female
85773,10.1117/12.2652626,Elisa López‐Varela,6.0,middle,False,2.609232e+07,ZA,True,ISGlobal,Elisa,female
85833,10.5588/pha.21.0072,E. Lopez Varela,9.0,middle,False,2.802694e+09,ZA,True,ISGlobal,E.,unknown


In [11]:
df_cerca.drop_duplicates('first_name').sort_values('first_name', ascending = False)

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center,first_name,gender
28291,10.1016/s2542-5196(21)00211-4,Érica Martínez-Solanas,2.0,middle,False,4.210148e+09,ES,True,ISGlobal,Érica,unknown
37901,10.1016/j.envres.2021.111392,Ángela Zumel-Marne,1.0,first,False,4.210155e+09,ES,True,ISGlobal,Ángela,female
39561,10.1016/j.mex.2024.102969,Á. Bach,3.0,middle,False,1.704866e+08,ES,True,ISGlobal,Á.,unknown
46466,10.1371/journal.ppat.1011557,Àngel Fenollar,5.0,middle,True,7.199913e+07,ES,True,ISGlobal,Àngel,unknown
46492,10.1093/ije/dyae063,Zoraida García,41.0,middle,False,1.704866e+08,ES,True,ISGlobal,Zoraida,female
...,...,...,...,...,...,...,...,...,...,...,...
33746,10.1002/jia2.25775,Adrià Murias‐Closas,2.0,middle,False,7.199913e+07,ES,True,ISGlobal,Adrià,unknown
41811,10.3390/cancers15102719,Adela Saco,2.0,middle,False,4.210115e+09,ES,True,ISGlobal,Adela,female
18263,10.1111/liv.14969,Adam Palayew,3.0,middle,False,2.801767e+09,CA,True,ISGlobal,Adam,male
24847,10.1186/s12879-023-08708-9,A. Bila,50.0,middle,False,4.210148e+09,ES,True,ISGlobal,A.,unknown


**We manually revise the classifier**

In [12]:
gender_check = g2d.download('1DtbOJzE9c9xCtuMfQQX8f4Uom6U7Mx7wwmzri57Xj4A', 'GenderISGlobal', col_names = True, row_names = False)
df_cerca = df_cerca.merge(gender_check[['first_name', 'gender_check']], on='first_name', how='left')
df_cerca['gender'] = df_cerca.apply(lambda row: row['gender_check'] if row['gender_check'] != '' else row['gender'], axis = 1)
df_cerca

Not all requested scopes were granted by the authorization server, missing scopes https://docs.google.com/feeds, https://spreadsheets.google.com/feeds.


,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center,first_name,gender,gender_check
0,10.1136/bmjopen-2021-051278,Carlos Chaccour,12.0,last,False,1.142924e+08,TZ,True,ISGlobal,Carlos,male,
1,10.3389/fphar.2021.625678,Carlos Chaccour,3.0,middle,False,1.142924e+08,TZ,True,ISGlobal,Carlos,male,
2,10.1371/journal.pntd.0009144,Carlos Chaccour,6.0,middle,True,1.142924e+08,TZ,True,ISGlobal,Carlos,male,
3,10.4269/ajtmh.20-1531,Carlos Chaccour,3.0,last,False,1.142924e+08,TZ,True,ISGlobal,Carlos,male,
4,10.1098/rstb.2019.0810,Carlos Chaccour,1.0,first,True,1.142924e+08,TZ,True,ISGlobal,Carlos,male,
...,...,...,...,...,...,...,...,...,...,...,...,...
16854,10.1002/ppul.25405,Elisa López‐Varela,15.0,last,False,1.380251e+08,ZA,True,ISGlobal,Elisa,female,
16855,10.1002/ppul.25405,Elisa López‐Varela,15.0,last,False,2.609232e+07,ZA,True,ISGlobal,Elisa,female,
16856,10.1117/12.2652626,Elisa López‐Varela,6.0,middle,False,2.609232e+07,ZA,True,ISGlobal,Elisa,female,
16857,10.5588/pha.21.0072,E. Lopez Varela,9.0,middle,False,2.802694e+09,ZA,True,ISGlobal,E.,unknown,


In [13]:
df_fem = df_cerca[df_cerca.gender.isin(['female'])]

df_fem.drop_duplicates(['DOI', 'Center']).groupby('Center').size() / \
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
ISGlobal    0.603175
dtype: float64

## \% publications led by women from the centre as authors

In [14]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.8163265306122449


**Percentage computed with the retrieved OA-raw_affiliation data identified author and not the whole**

In [15]:
df_fem_led = df_fem[(df_fem.author_position == 'first') | (df_fem.author_position == 'last') |(df_fem.is_corresponding == True) ]

df_fem_led.drop_duplicates(['DOI', 'Center']).groupby('Center').size() / \
df.drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
ISGlobal    0.277381
dtype: float64

## \% publications in collaboration with other CERCA centres

In [16]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.8163265306122449


In [17]:
df_tmp = pd.read_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_OA.csv')
df_tmp = df_tmp[df_tmp.Center != 'ISGlobal'].reset_index(drop = True)

df_tmp_2 = pd.concat((df_tmp, df)).drop_duplicates().reset_index(drop = True)
df_tmp_2.to_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_OA_v2.csv')
df_tmp_2

/tmp/ipykernel_7367/1717117934.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_tmp = pd.read_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_OA.csv')


,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center
0,10.1038/s41591-023-02610-2,José R. Banegas,88.0,middle,False,4.210155e+09,ES,False,ResearchMar
1,10.1038/s41591-023-02610-2,Charles Agyemang,47.0,middle,False,8.870644e+08,NL,False,ResearchMar
2,10.1038/s41591-023-02610-2,Andrew Wong,750.0,middle,False,4.512925e+07,GB,False,ResearchMar
3,10.1038/s41591-023-02610-2,Farhad Zamani,766.0,middle,False,1.611069e+08,IR,False,ResearchMar
4,10.1038/s41591-023-02610-2,José R. Banegas,88.0,middle,False,6.363444e+07,ES,False,ResearchMar
...,...,...,...,...,...,...,...,...,...
259288,10.1038/s41467-021-24673-w,Linda‐Gail Bekker,7.0,middle,False,1.380251e+08,ZA,False,ISGlobal
259289,10.1038/s41380-020-00976-0,Nastassja Koen,17.0,middle,False,1.576143e+08,ZA,False,ISGlobal
259290,10.1371/journal.pone.0248538,Fredros O. Okumu,8.0,last,False,1.926191e+08,ZA,False,ISGlobal
259291,10.1038/s41572-023-00452-3,Linda‐Gail Bekker,1.0,first,True,1.576143e+08,ZA,False,ISGlobal


In [18]:
institution = 'ISGlobal'

In [19]:
cerca_centers = {'BETA' : ['115304700'], # NOT IN OA - WE'LL USE THE PREVIOUS RESULT; I HAVE CONSIDERED THE ONE WITH MOST PUBLICATIONS
                    'CREAF' : ['4210129656'],
                    'ICN2' : ['4210093216'],
                    'ISGlobal' : ['4210148332'],
                    'ResearchMar' : ['4210156109']}

df_cerca_af = pd.read_csv('../data/external/ToCheck - AffID.csv')

df_center = df[(df.Center == institution) & (df.institution_id != int(cerca_centers[institution][0]))]
df_colab = df_center[df_center.institution_id.isin(df_cerca_af.OA_id)] 
percentage = df_colab.DOI.nunique() / df_center.DOI.nunique()
print(f"The percentage of publications in collaboration for {institution} is: {percentage:.2%}")

The percentage of publications in collaboration for ISGlobal is: 23.78%


In [ ]:
# df_unique = df_tmp_2[['DOI', 'Center', 'CERCA']].drop_duplicates()
# df_cerca = df_unique[df_unique['CERCA'] == True]
# dois_this = set(df_cerca.loc[df_cerca.Center == 'ISGlobal', 'DOI'])
# dois_others = set(df_cerca.loc[df_cerca.Center != 'ISGlobal', 'DOI'])
# collaborative_dois = dois_this & dois_others   
# percentage = len(collaborative_dois) / len(dois_this) if dois_this else 0
# print(f"The percentage of publications in collaboration for {institution} is: {percentage:.2%}")

The percentage of publications in collaboration for ISGlobal is: 10.45%


## \% publications in collaboration with other local institutions

In [20]:
print('The total percentage of publications analyzed is:', df[df.CERCA == True].DOI.nunique() / df_whole.DOI.nunique())

The total percentage of publications analyzed is: 0.8163265306122449


In [21]:
cerca_centers = {'BETA' : ['115304700'], # NOT IN OA - WE'LL USE THE PREVIOUS RESULT; I HAVE CONSIDERED THE ONE WITH MOST PUBLICATIONS
                    'CREAF' : ['4210129656'],
                    'ICN2' : ['4210093216'],
                    'ISGlobal' : ['4210148332'],
                    'ResearchMar' : ['4210156109']}

df_cerca_af = pd.read_csv('../data/external/ToCheck - AffID.csv')

df_center = df[(df.Center == institution) & (df.institution_id != int(cerca_centers[institution][0]))]
df_cerca_colab = df_center[df_center.institution_id.isin(df_cerca_af.OA_id)]
df_not_cerca_colab = df_center[(~df_center.DOI.isin(df_cerca_colab.DOI)) & (df_center.COUNTRY_CODE == 'ES')]
percentage = df_not_cerca_colab.DOI.nunique() / df_center.DOI.nunique()
print(f"The percentage of publications in collaboration for {institution} is: {percentage:.2%}")

The percentage of publications in collaboration for ISGlobal is: 58.69%


In [ ]:
# cerca_centers = {'BETA' : ['115304700'], # NOT IN OA - WE'LL USE THE PREVIOUS RESULT; I HAVE CONSIDERED THE ONE WITH MOST PUBLICATIONS
#                     'CREAF' : ['4210129656'],
#                     'ICN2' : ['4210093216'],
#                     'ISGlobal' : ['4210148332'],
#                     'ResearchMar' : ['4210156109']}

# df_cerca_af = pd.read_csv('../data/external/ToCheck - AffID.csv')

# for institution in df_cerca.Center.unique():
#     df_center = df[(df.Center == institution) & (df.institution_id != int(cerca_centers[institution][0]))]
#     df_cerca_colab = df_center[df_center.institution_id.isin(df_cerca_af.OA_id)] 
#     df_not_cerca_colab = df_center[(~df_center.DOI.isin(df_cerca_colab.DOI)) & (df_center.COUNTRY_CODE == 'ES')]
#     percentage = df_not_cerca_colab.DOI.nunique() / df_center.DOI.nunique()
#     print(f"The percentage of publications in collaboration for {institution} is: {percentage:.2%}")

In [ ]:
# df_unique = df_tmp_2[['DOI', 'Center', 'CERCA', 'COUNTRY_CODE']].drop_duplicates()
# df_cerca = df_unique[df_unique['CERCA'] == True]
# df_spanish_non_cerca = df_unique[(df_unique['CERCA'] == False) & (df_unique['COUNTRY_CODE'] == 'ES')]

# dois_center = set(df_cerca.loc[df_cerca['Center'] == 'ISGlobal', 'DOI'])
# dois_spanish_non_cerca = set(df_spanish_non_cerca['DOI'])
# collaborative_dois = dois_center & dois_spanish_non_cerca
# percentage = len(collaborative_dois) / len(dois_center)
# print(f'The percentage of publications analyzed for {institution} is: {percentage:.2%}')

The percentage of publications analyzed for ISGlobal is: 52.05%


## \% publications in collaboration with other international institutions

In [24]:
df_unique = df[['DOI', 'Center', 'CERCA', 'COUNTRY_CODE']].drop_duplicates()

df_cerca = df_unique[df_unique['CERCA'] == True]
df_international_non_cerca = df_unique[(df_unique['CERCA'] == False) & (df_unique['COUNTRY_CODE'] != 'ES')]

dois_center = set(df_cerca.loc[df_cerca['Center'] == institution, 'DOI'])
dois_international = set(df_international_non_cerca['DOI'])

collaborative_dois = dois_center & dois_international
    
percentage = len(collaborative_dois) / len(dois_center) if dois_center else 0
print(f'The percentage of international collaborations for {institution} is: {percentage:.2%}')

The percentage of international collaborations for ISGlobal is: 80.40%
